In [1]:
# ============================================================
# SECURESENSE - GROUP-AWARE TRAIN / TEST SPLIT
# ============================================================
#
# Purpose:
#   Create a leakage-resistant split where complete source files
#   are kept entirely in either TRAIN or TEST.
#
# IMPORTANT:
#   _source_file is used ONLY for grouping.
#   It is NEVER used as a model feature.
# ============================================================

import os
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit


# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

RANDOM_STATE = 42

DATA_PATH = "../data/cleaned/securesense_final_binary.parquet"

OUTPUT_DIR = "../data/group_splits"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ------------------------------------------------------------
# 1. LOAD CLEANED DATASET
# ------------------------------------------------------------

print("=" * 80)
print("SECURESENSE - GROUP-AWARE DATA SPLIT")
print("=" * 80)

print("\nLoading cleaned dataset...")

df = pd.read_parquet(DATA_PATH)

print(f"Total rows: {len(df):,}")
print(f"Total columns: {len(df.columns):,}")


# ------------------------------------------------------------
# 2. CHECK SOURCE FILE INFORMATION
# ------------------------------------------------------------

if "_source_file" not in df.columns:
    raise ValueError(
        "_source_file column is missing. "
        "Cannot perform group-aware split."
    )

print("\nSource files:")
print(df["_source_file"].value_counts())


# ------------------------------------------------------------
# 3. DEFINE MODEL FEATURES
# ------------------------------------------------------------

EXCLUDED_COLUMNS = [
    "BinaryLabel",
    "Label",
    "_source_file"
]

FEATURE_COLUMNS = [
    column
    for column in df.columns
    if column not in EXCLUDED_COLUMNS
]

X = df[FEATURE_COLUMNS].copy()

y = df["BinaryLabel"].map({
    "BENIGN": 0,
    "ATTACK": 1
})

if y.isna().any():
    raise ValueError("Unexpected BinaryLabel value found.")

y = y.astype(np.int8)

groups = df["_source_file"].copy()


print("\nModel feature information:")
print(f"Number of features: {len(FEATURE_COLUMNS)}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")


# ------------------------------------------------------------
# 4. GROUP-AWARE SPLIT
# ------------------------------------------------------------
#
# Entire source files stay together.
#
# No source file can appear in both TRAIN and TEST.
#
# This prevents the model from learning patterns from one
# source file and being tested on rows from that same file.
# ------------------------------------------------------------

print("\nCreating group-aware split...")

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups=groups
    )
)


# ------------------------------------------------------------
# 5. CREATE TRAIN / TEST DATAFRAMES
# ------------------------------------------------------------

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)


# ------------------------------------------------------------
# 6. REMOVE NON-MODEL COLUMNS
# ------------------------------------------------------------

# Keep source file temporarily for validation and analysis.
#
# We will remove it from X when training the models.

train_features = train_df[FEATURE_COLUMNS].copy()
test_features = test_df[FEATURE_COLUMNS].copy()

train_labels = train_df["BinaryLabel"].map({
    "BENIGN": 0,
    "ATTACK": 1
}).astype(np.int8)

test_labels = test_df["BinaryLabel"].map({
    "BENIGN": 0,
    "ATTACK": 1
}).astype(np.int8)


# ------------------------------------------------------------
# 7. CHECK SOURCE-FILE LEAKAGE
# ------------------------------------------------------------

train_sources = set(
    train_df["_source_file"].unique()
)

test_sources = set(
    test_df["_source_file"].unique()
)

overlap = train_sources.intersection(
    test_sources
)

print("\n" + "=" * 80)
print("SOURCE-FILE LEAKAGE CHECK")
print("=" * 80)

print("\nTraining source files:")
for source in sorted(train_sources):
    print("  ", source)

print("\nTesting source files:")
for source in sorted(test_sources):
    print("  ", source)

print("\nSource files appearing in BOTH:")
print(overlap)

if len(overlap) != 0:
    raise RuntimeError(
        "FAILED: Source-file leakage detected!"
    )

print("\nPASS: No source file appears in both TRAIN and TEST.")


# ------------------------------------------------------------
# 8. CLASS DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CLASS DISTRIBUTION")
print("=" * 80)

print("\nTRAIN:")
print(
    train_labels.value_counts()
    .rename(index={
        0: "BENIGN",
        1: "ATTACK"
    })
)

print("\nTRAIN percentages:")
print(
    train_labels.value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename(index={
        0: "BENIGN",
        1: "ATTACK"
    })
)

print("\nTEST:")
print(
    test_labels.value_counts()
    .rename(index={
        0: "BENIGN",
        1: "ATTACK"
    })
)

print("\nTEST percentages:")
print(
    test_labels.value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename(index={
        0: "BENIGN",
        1: "ATTACK"
    })
)


# ------------------------------------------------------------
# 9. SOURCE FILE DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TRAIN SOURCE DISTRIBUTION")
print("=" * 80)

print(
    pd.crosstab(
        train_df["_source_file"],
        train_df["BinaryLabel"]
    )
)

print("\n" + "=" * 80)
print("TEST SOURCE DISTRIBUTION")
print("=" * 80)

print(
    pd.crosstab(
        test_df["_source_file"],
        test_df["BinaryLabel"]
    )
)


# ------------------------------------------------------------
# 10. SAVE TRAIN / TEST DATA
# ------------------------------------------------------------

train_path = os.path.join(
    OUTPUT_DIR,
    "group_train.parquet"
)

test_path = os.path.join(
    OUTPUT_DIR,
    "group_test.parquet"
)

train_df.to_parquet(
    train_path,
    index=False
)

test_df.to_parquet(
    test_path,
    index=False
)


# ------------------------------------------------------------
# 11. FINAL VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL VALIDATION")
print("=" * 80)

print(f"\nTraining rows: {len(train_df):,}")
print(f"Testing rows : {len(test_df):,}")

print(f"\nTraining features: {train_features.shape}")
print(f"Testing features : {test_features.shape}")

print(
    f"\nMissing values - TRAIN: "
    f"{train_features.isna().sum().sum():,}"
)

print(
    f"Missing values - TEST : "
    f"{test_features.isna().sum().sum():,}"
)

print(
    f"\nInfinite values - TRAIN: "
    f"{np.isinf(train_features.select_dtypes(include=np.number)).sum().sum():,}"
)

print(
    f"Infinite values - TEST : "
    f"{np.isinf(test_features.select_dtypes(include=np.number)).sum().sum():,}"
)


# ------------------------------------------------------------
# 12. SAVE FEATURE LIST
# ------------------------------------------------------------

feature_path = os.path.join(
    OUTPUT_DIR,
    "feature_columns.txt"
)

with open(feature_path, "w") as f:
    for feature in FEATURE_COLUMNS:
        f.write(feature + "\n")


# ------------------------------------------------------------
# 13. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("GROUP-AWARE SPLIT COMPLETE")
print("=" * 80)

print("\nFiles saved:")

print(
    f"TRAIN: {train_path}"
)

print(
    f"TEST : {test_path}"
)

print(
    f"FEATURES: {feature_path}"
)

print("\nSource-file overlap:", len(overlap))

if len(overlap) == 0:
    print("\n✓ GROUP LEAKAGE CHECK PASSED")

print("\n✓ Dataset ready for group-aware model evaluation.")
print("=" * 80)

SECURESENSE - GROUP-AWARE DATA SPLIT

Loading cleaned dataset...
Total rows: 2,231,266
Total columns: 72

Source files:
_source_file
DoS-Wednesday-no-metadata.parquet            563660
Benign-Monday-no-metadata.parquet            458826
Bruteforce-Tuesday-no-metadata.parquet       383085
DDoS-Friday-no-metadata.parquet              216498
Infiltration-Thursday-no-metadata.parquet    189769
Botnet-Friday-no-metadata.parquet            174776
WebAttacks-Thursday-no-metadata.parquet      138561
Portscan-Friday-no-metadata.parquet          106091
Name: count, dtype: int64

Model feature information:
Number of features: 69
X shape: (2231266, 69)
y shape: (2231266,)

Creating group-aware split...

SOURCE-FILE LEAKAGE CHECK

Training source files:
   Benign-Monday-no-metadata.parquet
   Bruteforce-Tuesday-no-metadata.parquet
   DDoS-Friday-no-metadata.parquet
   DoS-Wednesday-no-metadata.parquet
   Portscan-Friday-no-metadata.parquet
   WebAttacks-Thursday-no-metadata.parquet

Testing source 

In [3]:
import pandas as pd
import json

TEST_PATH = r"W:\FinalYearProject\SecureSense_Project\notebooks\data\final_group_splits\test.parquet"

df = pd.read_parquet(TEST_PATH)

# Take one real test row
row = df.drop(
    columns=["BinaryLabel", "Label", "_source_file"],
    errors="ignore"
).iloc[0]

# Convert to normal Python values
data = row.to_dict()

print("Number of features:", len(data))
print("\nExpected features:", len(row.index))

print("\nOriginal label of this test row:")
print(df.iloc[0]["BinaryLabel"])

print("\nJSON ready for /predict:")
print(json.dumps(data, indent=2))

Number of features: 69

Expected features: 69

Original label of this test row:
BENIGN

JSON ready for /predict:
{
  "Protocol": 6.0,
  "Flow Duration": 4.0,
  "Total Fwd Packets": 2.0,
  "Total Backward Packets": 0.0,
  "Fwd Packets Length Total": 12.0,
  "Bwd Packets Length Total": 0.0,
  "Fwd Packet Length Max": 6.0,
  "Fwd Packet Length Min": 6.0,
  "Fwd Packet Length Mean": 6.0,
  "Fwd Packet Length Std": 0.0,
  "Bwd Packet Length Max": 0.0,
  "Bwd Packet Length Min": 0.0,
  "Bwd Packet Length Mean": 0.0,
  "Bwd Packet Length Std": 0.0,
  "Flow Bytes/s": 3000000.0,
  "Flow Packets/s": 500000.0,
  "Flow IAT Mean": 4.0,
  "Flow IAT Std": 0.0,
  "Flow IAT Max": 4.0,
  "Flow IAT Min": 4.0,
  "Fwd IAT Total": 4.0,
  "Fwd IAT Mean": 4.0,
  "Fwd IAT Std": 0.0,
  "Fwd IAT Max": 4.0,
  "Fwd IAT Min": 4.0,
  "Bwd IAT Total": 0.0,
  "Bwd IAT Mean": 0.0,
  "Bwd IAT Std": 0.0,
  "Bwd IAT Max": 0.0,
  "Bwd IAT Min": 0.0,
  "Fwd PSH Flags": 0.0,
  "Fwd URG Flags": 0.0,
  "Fwd Header Length": 40.

In [4]:
import pandas as pd

TEST_PATH = r"W:\FinalYearProject\SecureSense_Project\notebooks\data\final_group_splits\test.parquet"

df = pd.read_parquet(TEST_PATH)

# Take 100 real test flows
sample = df.head(100).copy()

# Remove target and source metadata
sample = sample.drop(
    columns=["BinaryLabel", "Label", "_source_file"],
    errors="ignore"
)

CSV_PATH = r"W:\FinalYearProject\SecureSense_Project\test_prediction.csv"

sample.to_csv(
    CSV_PATH,
    index=False
)

print("CSV created successfully!")
print("Rows:", len(sample))
print("Features:", sample.shape[1])
print("Path:", CSV_PATH)

CSV created successfully!
Rows: 100
Features: 69
Path: W:\FinalYearProject\SecureSense_Project\test_prediction.csv


In [5]:
import pandas as pd

TEST_PATH = r"W:\FinalYearProject\SecureSense_Project\notebooks\data\final_group_splits\test.parquet"

df = pd.read_parquet(TEST_PATH)

# Select only known ATTACK rows
attack_df = df[df["BinaryLabel"] == "ATTACK"].copy()

print("Total attack rows available:", len(attack_df))

# Take first 50 known attack flows
attack_sample = attack_df.head(50).copy()

# Remove target and metadata columns
attack_sample = attack_sample.drop(
    columns=["BinaryLabel", "Label", "_source_file"],
    errors="ignore"
)

ATTACK_CSV_PATH = r"W:\FinalYearProject\SecureSense_Project\attack_test_50.csv"

attack_sample.to_csv(
    ATTACK_CSV_PATH,
    index=False
)

print("\nAttack CSV created successfully!")
print("Rows:", len(attack_sample))
print("Features:", attack_sample.shape[1])
print("Path:", ATTACK_CSV_PATH)

Total attack rows available: 128011

Attack CSV created successfully!
Rows: 50
Features: 69
Path: W:\FinalYearProject\SecureSense_Project\attack_test_50.csv


In [2]:
print("TRAIN SHAPE:", train_df.shape)
print("TEST SHAPE :", test_df.shape)

print("\nTRAIN LABELS:")
print(train_df["BinaryLabel"].value_counts(normalize=True) * 100)

print("\nTEST LABELS:")
print(test_df["BinaryLabel"].value_counts(normalize=True) * 100)

print("\nChecking duplicate feature rows between TRAIN and TEST...")

feature_cols = [
    c for c in train_df.columns
    if c != "BinaryLabel"
]

train_keys = pd.util.hash_pandas_object(
    train_df[feature_cols],
    index=False
)

test_keys = pd.util.hash_pandas_object(
    test_df[feature_cols],
    index=False
)

overlap = len(set(train_keys) & set(test_keys))

print("Exact feature-row overlap:", overlap)

TRAIN SHAPE: (1866721, 72)
TEST SHAPE : (364545, 72)

TRAIN LABELS:
BinaryLabel
BENIGN    82.066201
ATTACK    17.933799
Name: proportion, dtype: float64

TEST LABELS:
BinaryLabel
BENIGN    99.602793
ATTACK     0.397207
Name: proportion, dtype: float64

Checking duplicate feature rows between TRAIN and TEST...
Exact feature-row overlap: 0
